# 后端切换总结与常见问题

回顾前面的实践，训练后端切换可以整理为六步：确定切换范围、选择引擎、映射配置、接通计算接口、接通状态接口、运行训练。下面沿着这六步串起关键配置和日志。


## 切换过程回顾

| 步骤 | Wordle 训练中的做法 |
| --- | --- |
| 1. 确定范围 | 模型、数据、GRPO、vLLM、AgentLoop 和奖励保持不变；Actor/Ref 后端变化 |
| 2. 选择引擎 | `model_engine=torchtitan` 让 Verl 创建 TorchTitan Engine |
| 3. 映射配置 | FSDP 配置域映射为 DeviceMesh、FSDP2、offload 与 reshard；训练精度采用 TorchTitan 默认值 |
| 4. 接通计算接口 | packed token 通过 VarlenMetadata 进入 TND 变长注意力，计算后恢复 jagged tensor |
| 5. 接通状态接口 | TorchTitan 完成 Actor/Ref 计算，并通过分片状态接口支持 Actor 权重同步 |
| 6. 运行训练 | DRY_RUN 先核对启动参数，再用 3 个训练 step 检查整条数据流 |

当前两卡配置使用 DP shard 2、CP 1，并在训练计算中启用 TND 变长注意力。CP、TP、PP 和 EP 也可以与 TorchTitan-NPU 组合，但需要根据模型规模、序列长度和硬件拓扑重新规划并行维度，三步训练暂不启用。


## 常见问题

| 现象 | 可能原因 | 处理方式 |
| --- | --- | --- |
| 环境准备脚本中止 | Python、GCC、CANN、uv、网络或双 NPU 条件不满足 | 根据脚本最后一条明确错误修复对应前置条件，再重新执行环境准备 |
| ModelScope 下载中断 | 网络波动、访问限制或磁盘空间不足 | 保留已下载文件，修复网络或磁盘问题后重新执行同一 Cell |
| Wordle 数据生成失败 | TextArena、NLTK、pandas 或 pyarrow 不可用 | 检查 `setup_backend.sh` 的依赖安装结果，再重新执行资产准备 |
| 训练资产完整性检查失败 | 模型缺少 `config.json` 或 safetensors，或 parquet 不完整 | 回到 07.02 补齐资产，再继续确认启动命令 |
| DRY_RUN 无法完成 | 独立环境、训练资产或 launcher 版本异常 | 回到环境与资产准备阶段确认最后结果 |
| DRY_RUN 缺少 TorchTitan、FSDP2 或 TND 配置 | 使用了错误入口或训练代码版本 | 确认运行 `torchtitan_backend` launcher，并检查输出中的后端、DP shard、CP 和 attention 配置 |
| TND 前向失败 | packed 样本边界元数据、有效 token 或长度契约不一致 | 从 Actor worker 首个异常核对 VarlenMetadata、offsets 和 5120 单样本上限 |
| rollout 阶段失败 | vLLM、生成长度、AgentLoop 或训练资产异常 | 从 rollout 日志中的首个异常定位生成与交互边界 |
| Actor update 阶段失败 | FSDP2、NPU 算子、offload 或训练 batch 执行异常 | 保持课程训练负载，从 Actor worker 的首个异常定位训练后端 |
| 三步训练完成但 reward 波动 | 训练步数较少，样本差异会直接反映在 reward 上 | 先确认三步数据流完整；观察训练趋势时再增加步数 |

排查时先找到最后一个成功阶段。例如，rollout 已经结束而 Actor update 尚未完成，就可以先把注意力放在训练 worker，而不必重新检查数据下载。保留相同的 batch、长度和 rollout 配置，也便于复现问题。


## 如何观察性能变化

一个 RL step 包含 vLLM rollout、奖励、old log-prob、Ref log-prob、Actor update 和权重同步。Wordle 的交互轮数和生成长度会变化，`timing_s/step` 往往先受到 rollout 影响。比较训练后端时，除了总耗时，还要拆开看：

- `timing_s/update_actor`：Actor 前向、反向和优化器更新耗时；
- `timing_per_token_ms/update_actor`：按有效 token 数归一化后的 Actor 耗时；
- `timing_s/old_log_prob` 与 `timing_s/ref`：训练后端的两条前向路径；
- 峰值显存与可支持的 packed token 容量：配置能否承载更大的 batch 或更长的序列。

### 课程开发阶段的测试结果

下面的数据来自单机两卡环境。三组测试都使用 Qwen3-1.7B、同一份 Wordle 数据、seed 42、`train_batch_size=128`、`rollout.n=8`、4096 token 最大响应长度和六轮 AgentLoop，并分别运行 3 个 step。为了减少额外耗时，测试关闭了 checkpoint 保存和训练结束后的验证。表中的“每卡吞吐”对应 Verl 的 `perf/throughput`，计算方式是总 token 数除以单步耗时和卡数。

| 训练路径 | 平均单步耗时 | Actor update | Actor 每 token 耗时 | 每卡吞吐 |
| --- | ---: | ---: | ---: | ---: |
| 初阶 Verl FSDP | 703.9 s | 244.5 s | 0.1252 ms | 1387 token/s |
| TorchTitan FSDP2 + TND（DP shard 2，CP 1） | 670.3 s | 198.7 s | 0.1020 ms | 1453 token/s |
| TorchTitan TND + CP2（DP shard 1，CP 2） | 693.6 s | 215.4 s | 0.1122 ms | 1384 token/s |

这次运行中，FSDP2 + TND 的 Actor 每 token 耗时比初阶路径低约 18.5%，平均单步耗时低约 4.8%。总耗时的变化小得多，也说明 rollout 在整个 step 中占有较大比重。改成 CP2 后，Actor 每 token 耗时比 DP shard 2、CP 1 高约 10.0%；对于当前模型和长度配置，把两张卡用于 FSDP2 更合适。

读取这组数据时还要注意两个变量：初阶路径切换到 FSDP2 + TND 时，训练引擎和注意力实现同时发生了变化；两卡启用 CP2 时，DP shard 也从 2 变成了 1。因此，表格适合帮助理解各阶段耗时，不能单独拆出 TorchTitan、TND 或 CP 各自带来的变化。若要做正式对比，需要一次只调整一个条件，增加预热和重复次数，再报告平均值和波动范围。


## 完成本课程后

现在，你应该能够说清训练后端切换涉及哪些模块，解释 FSDP2、offload 与 TND 怎样进入 Wordle RL 流程，使用 DRY_RUN 检查启动参数，并独立完成 3 个训练 step。继续开展长期训练时，可以再根据任务需要加入 checkpoint、断点续训、周期验证和更严格的性能测试。


## 课后练习

### 判断题

1. （判断题）启用 CPU offload 后，Actor 参数和优化器状态在计算期间也始终停留在主机内存。

2. （判断题）比较训练后端性能时，只看包含 vLLM rollout 的总 step 耗时即可。

### 单选题

3. （单选题）当长序列的激活或注意力显存成为主要瓶颈时，本课程介绍的哪项能力主要用于扩展序列维度容量？

   A. Context Parallel

   B. Wordle 奖励函数

   C. checkpoint 恢复

   D. 优化器学习率衰减

### 多选题

4. （多选题）评价训练后端的训练阶段性能时，应优先关注哪些指标？

   A. `timing_s/update_actor`

   B. `timing_per_token_ms/update_actor`

   C. `timing_s/old_log_prob` 与 `timing_s/ref`

   D. 只比较单次 `timing_s/step`

5. （多选题）根据失败阶段进行定位时，下列哪些对应关系合理？

   A. 环境准备失败检查软件版本、网络与磁盘

   B. TND 前向失败检查 offsets 与 VarlenMetadata

   C. rollout 失败检查 vLLM 与 AgentLoop

   D. Actor update 失败检查训练后端


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/08_validation_and_troubleshooting/answer/08.02_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
